# Fit on train, score on train

Open `day1/models.py`. Each pipeline starts with scaling for the numeric columns and one-hot encoding for district, plan, and aisle. The classifier is the last step. `cross_val_score` refits inside the train rows only. Leave the validation rows alone until the next notebook.

A pipeline fits the scaler on the same rows as the classifier. That is what keeps the column averages from seeing the test rows.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    if (candidate / "day1" / "schema.py").is_file() and (candidate / "app.py").is_file():
        root = candidate
        break
    nested = candidate / "01-supervised-churn"
    if (nested / "day1" / "schema.py").is_file() and (nested / "app.py").is_file():
        root = nested
        break
if root is None:
    raise FileNotFoundError(
        "Start Jupyter from the repo folder, from 01-supervised-churn, or from its notebooks folder."
    )
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
ROOT = root
print("cwd", here)
print("module", ROOT)


In [ ]:
import pandas as pd

customers = pd.read_csv(ROOT / "data" / "klongcart_churn.csv")
print(customers.shape)
customers.head()


In [ ]:
from day1.frame import features_and_target
from day1.split import split_customers

train, validation, test = split_customers(customers)
print("train", len(train), "validation", len(validation), "test", len(test))
X_train, y_train = features_and_target(train)
X_val, y_val = features_and_target(validation)
X_test, y_test = features_and_target(test)
print("churned is a label, not a feature:", "churned" not in X_train.columns)


In [ ]:
from sklearn.model_selection import cross_val_score

from day1.models import make_logistic_pipeline

logistic = make_logistic_pipeline()
logistic.fit(X_train, y_train)
logistic_cv = cross_val_score(
    logistic, X_train, y_train, cv=3, scoring="roc_auc", n_jobs=1
)
print("logistic cv mean", round(float(logistic_cv.mean()), 3))


### Exercises

Fit a forest pipeline the same way. `make_forest_pipeline` is in `day1.models`. Use `cross_val_score` on `X_train` and `y_train` only, with `cv=3` and `scoring="roc_auc"`. Store the mean in `forest_cv_mean`.

Then score the fitted logistic model on the validation rows. `roc_auc_score` needs the labels and column 1 of `predict_proba`. Store that number in `logistic_val_auc`.


In [ ]:
from sklearn.metrics import roc_auc_score

from day1.models import make_forest_pipeline

forest = None
forest_cv_mean = None
logistic_val_auc = None


In [ ]:
if forest_cv_mean is None or logistic_val_auc is None:
    raise NotImplementedError("Finish the forest CV exercise and the logistic validation AUC.")
print("forest cv mean", round(float(forest_cv_mean), 3))
print("logistic validation AUC", round(float(logistic_val_auc), 3))


## Checkpoint

Cross-validation here is only a train-set check. Notebook 04 compares models on the validation rows. If you are behind at 13:35, skip the forest exercise, keep the logistic printout, and move on.
